# 🎵 T3 Triple Oscillator

Three coupled Kuramoto populations with Stuart-Landau amplitude dynamics:

$$\dot{\theta}_i = \omega_i + \sum_j W_{ij} \sin(\theta_j - \theta_i) + D_i$$

$$\dot{r}_i = \lambda_i r_i - \beta_i r_i^3 + \text{drive}$$

Each population has a different frequency band and they interact via cross-coupling.

**Applications:** EEG generation, phase coding, feature extraction for reservoir computing.


## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import taas_checkmate as ne
except ImportError:
    import subprocess
    subprocess.run(["maturin", "develop", "--release", "-q"], check=True)
    import taas_checkmate as ne

plt.style.use("dark_background")
N = 64
N_PAD = (N + 7) // 8 * 8
DT = 0.5

print("✅ Setup complete")


## Create T3 Oscillator

In [ ]:
# Create T3 with 3 populations at different frequency ranges
tri = ne.PyTriOscillator(N_PAD, freq_min=2.0, freq_max=80.0)
tri.set_uniform_coupling(0.5, 0.3, 0.2)  # intra, cross1, cross2

print(f"T3: {tri.n_osc()} oscillators × 3 populations")
print(f"  Population 1: theta/alpha  (2-8 Hz)")
print(f"  Population 2: beta         (8-30 Hz)")
print(f"  Population 3: gamma        (30-80 Hz)")
print(f"  Coupling: intra={0.5}, cross={0.3}/{0.2}")

# Feature dimension
dim = ne.pst_linear_dim(tri.n_osc())
print(f"  PST feature dimension: {dim}")


## Run T3 and Visualize Dynamics

In [ ]:
STEPS = 2000
phase_out = np.zeros(N_PAD, dtype=np.float32)
amp_out = np.zeros(N_PAD, dtype=np.float32)

hist = {
    "ph1": np.zeros((STEPS, min(N, 10)), dtype=np.float32),
    "ph2": np.zeros((STEPS, min(N, 10)), dtype=np.float32),
    "ph3": np.zeros((STEPS, min(N, 10)), dtype=np.float32),
    "r1": np.zeros(STEPS, dtype=np.float32),
    "r2": np.zeros(STEPS, dtype=np.float32),
    "r3": np.zeros(STEPS, dtype=np.float32),
    "order1": np.zeros(STEPS, dtype=np.float32),
    "order2": np.zeros(STEPS, dtype=np.float32),
    "order3": np.zeros(STEPS, dtype=np.float32),
}

from rich.progress import Progress

with Progress() as pbar:
    task = pbar.add_task("[cyan]Simulating...", total=STEPS)
    for t in range(STEPS):
        tri.step(DT)

        for pop in range(1, 4):
            tri.copy_phases(phase_out, pop)
            tri.copy_amplitudes(amp_out, pop)
            if pop == 1:
                hist["ph1"][t] = phase_out[:10]
                hist["r1"][t] = np.mean(amp_out[:N])
                hist["order1"][t] = tri.order_parameter_r1()
            elif pop == 2:
                hist["ph2"][t] = phase_out[:10]
                hist["r2"][t] = np.mean(amp_out[:N])
                hist["order2"][t] = tri.order_parameter_r2()
            else:
                hist["ph3"][t] = phase_out[:10]
                hist["r3"][t] = np.mean(amp_out[:N])
                hist["order3"][t] = tri.order_parameter_r3()

        pbar.update(task, advance=1)

print("Simulation complete!")


## Population Amplitudes & Order Parameters

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7))

t = np.arange(STEPS) * DT

# Amplitudes
ax1.plot(t, hist["r1"], color="#ff6600", linewidth=0.8, label="Pop 1 (θ/α)")
ax1.plot(t, hist["r2"], color="#3399ff", linewidth=0.8, label="Pop 2 (β)")
ax1.plot(t, hist["r3"], color="#ff33cc", linewidth=0.8, label="Pop 3 (γ)")
ax1.set_ylabel("Mean Amplitude")
ax1.set_title("Stuart-Landau Amplitudes")
ax1.legend()
ax1.grid(alpha=0.1)

# Order parameters
ax2.plot(t, hist["order1"], color="#ff6600", linewidth=0.8, label="Pop 1")
ax2.plot(t, hist["order2"], color="#3399ff", linewidth=0.8, label="Pop 2")
ax2.plot(t, hist["order3"], color="#ff33cc", linewidth=0.8, label="Pop 3")
ax2.set_xlabel("Time (ms)")
ax2.set_ylabel("Order Parameter r")
ax2.set_title("Synchronization (1.0 = perfect sync)")
ax2.legend()
ax2.grid(alpha=0.1)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.show()


## Phase Portraits

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), subplot_kw={"projection": "polar"})

for ax, ph, pop, color in zip(
    axes,
    [hist["ph1"], hist["ph2"], hist["ph3"]],
    ["Pop 1 (θ/α)", "Pop 2 (β)", "Pop 3 (γ)"],
    ["#ff6600", "#3399ff", "#ff33cc"],
):
    # Plot phase trajectory of first oscillator
    ax.plot(ph[:, 0], np.arange(STEPS) * 0.01, alpha=0, linewidth=0)
    ax.scatter(ph[:, 0], np.ones(STEPS), s=1, c=color, alpha=0.3)
    ax.set_title(pop, color=color)
    ax.grid(alpha=0.1)

plt.suptitle("Phase Evolution on Unit Circle", fontsize=14)
plt.tight_layout()
plt.show()


## PST Feature Extraction

In [ ]:
# Compute PST features for readout
fv = ne.pst_linear_features(tri)
print(f"Feature vector dimension: {len(fv)}")
print(f"  Min: {fv.min():.4f}, Max: {fv.max():.4f}, Mean: {fv.mean():.4f}")

# Features over time
n_features = dim
feature_history = np.zeros((STEPS, n_features), dtype=np.float32)

for t in range(STEPS):
    tri.step(DT)
    fv = ne.pst_linear_features(tri)
    feature_history[t] = fv[:n_features]

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(feature_history.T, aspect="auto", cmap="RdBu_r",
               extent=[0, STEPS * DT, 0, n_features])
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Feature Index")
ax.set_title("PST Feature Timeline")
plt.colorbar(im, ax=ax, label="Feature Value", shrink=0.8)
plt.tight_layout()
plt.show()


## EEG from T3

In [ ]:
# Mix T3 populations into EEG-like signal
eeg = hist["r1"] * np.sin(2 * np.pi * 5 * t / 1000) + \
      hist["r2"] * np.sin(2 * np.pi * 15 * t / 1000) + \
      hist["r3"] * np.sin(2 * np.pi * 45 * t / 1000)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6))

ax1.plot(t[:500], eeg[:500], color="#00ff88", linewidth=0.6)
ax1.set_xlabel("Time (ms)")
ax1.set_ylabel("EEG (μV)")
ax1.set_title("T3-Derived EEG Signal (500ms window)")
ax1.grid(alpha=0.1)

freqs = np.fft.rfftfreq(STEPS, d=DT / 1000)
psd = np.abs(np.fft.rfft(eeg * np.hanning(STEPS))) ** 2
ax2.semilogy(freqs, psd, color="#00ff88", linewidth=0.8)
ax2.axvspan(2, 8, alpha=0.15, color="#ff6600", label="θ/α")
ax2.axvspan(8, 30, alpha=0.15, color="#3399ff", label="β")
ax2.axvspan(30, 80, alpha=0.15, color="#ff33cc", label="γ")
ax2.set_xlabel("Frequency (Hz)")
ax2.set_ylabel("PSD")
ax2.set_title("EEG Power Spectrum")
ax2.set_xlim(0, 100)
ax2.legend()
ax2.grid(alpha=0.1)

plt.tight_layout()
plt.show()
